# Notebook 01 — Exploratory Data Analysis (EDA)

MIQR-CC ERCP Classification — Universidade do Minho, 2025/2026

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.paths import load_config, ensure_dirs
from src.utils.seed import set_seed
from src.data.eda import count_images, get_image_sizes, get_sample_paths, run_eda
from src.utils.plots import plot_class_distribution, plot_sample_images
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

cfg = load_config('../config.yaml')
set_seed(cfg['seed'])
ensure_dirs(cfg)
CLASS_NAMES = cfg['data']['class_names']
print('Config loaded. Classes:', CLASS_NAMES)

## 1. Class distribution across splits

In [ ]:
for split in ['train', 'val', 'test']:
    counts = count_images(cfg['data'][f'{split}_dir'], CLASS_NAMES)
    total = sum(counts.values())
    print(f'\n{split.upper()} ({total} images):')
    for cls, n in counts.items():
        bar = '█' * (n // 10)
        print(f'  {cls:<20} {n:>4}  {bar}')

    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(counts.keys(), counts.values(), color=['#e74c3c','#3498db','#2ecc71','#f39c12'])
    ax.set_title(f'Class distribution — {split.capitalize()}')
    ax.set_ylabel('Images')
    for i, v in enumerate(counts.values()):
        ax.text(i, v+1, str(v), ha='center')
    plt.tight_layout()
    plt.savefig(f'../outputs/figures/class_distribution_{split}.png', dpi=150)
    plt.show()

## 2. Image size distribution

In [ ]:
import numpy as np
sizes = get_image_sizes(cfg['data']['train_dir'], CLASS_NAMES, max_per_class=150)

if sizes:
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
    print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
    ax1.hist(widths, bins=30, color='#3498db', edgecolor='black')
    ax1.set_title('Image Width Distribution')
    ax2.hist(heights, bins=30, color='#e74c3c', edgecolor='black')
    ax2.set_title('Image Height Distribution')
    plt.tight_layout()
    plt.savefig('../outputs/figures/image_size_distribution.png', dpi=150)
    plt.show()
else:
    print('No images found — place dataset in data/processed/ first.')

## 3. Sample images per class

In [ ]:
from PIL import Image
samples = get_sample_paths(cfg['data']['train_dir'], CLASS_NAMES, n=4)

fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(14, len(CLASS_NAMES)*3))
for row, cls in enumerate(CLASS_NAMES):
    paths = samples.get(cls, [])
    for col in range(4):
        ax = axes[row][col]
        if col < len(paths):
            img = Image.open(paths[col]).convert('RGB')
            ax.imshow(img)
            if col == 0:
                ax.set_ylabel(cls, fontsize=10, fontweight='bold')
        ax.axis('off')
plt.suptitle('Sample Images per Class (Train)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/sample_images_per_class.png', dpi=150)
plt.show()

## 4. Class imbalance analysis

In [ ]:
from src.data.dataset import ERCPDataset
from src.data.transforms import get_transforms

train_ds = ERCPDataset(cfg['data']['train_dir'],
                       transform=get_transforms(224, 'train'),
                       class_names=CLASS_NAMES)
counts = train_ds.get_class_counts()
weights = train_ds.get_class_weights()

print('Class counts:', counts)
print('Class weights (for loss):')
for cls, w in zip(CLASS_NAMES, weights):
    print(f'  {cls:<20}: {w:.4f}')